# KOCOH Baseline 

In [ ]:
!pip install pandas scikit-learn transformers torch -q

## 1. 라이브러리 및 기본 설정

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import copy
import gc
from sklearn.utils.class_weight import compute_class_weight

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42
MAX_LENGTH = 128
BATCH_SIZE = 4
EPOCHS = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)
print("EPOCHS:", EPOCHS)
print("BATCH_SIZE:", BATCH_SIZE)

## 2. KOCOH 데이터 불러오기

In [ ]:
TRAIN_DATA_PATH = '../src/team_train_context_split.csv'
VALID_DATA_PATH = '../src/team_valid_context_split.csv'
TEST_DATA_PATH = '../src/team_test_context_split.csv'

RESULT_DIR = 'result'
RESULT_FILE = os.path.join(RESULT_DIR, 'final_context_split_results.csv')
os.makedirs(RESULT_DIR, exist_ok=True)

## 3. Dataset 클래스와 평가 함수

In [ ]:
class HateSpeechDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def evaluate_model(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")

            outputs = model(**batch, labels=labels)
            total_loss += outputs.loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "y_true": all_labels,
        "y_pred": all_preds
    }

## 4. Baseline 학습 함수

In [ ]:
def train_baseline_model(model_name, output_name, use_context=False):
    print("=" * 70)
    print("학습 모델:", model_name)
    print("=" * 70)

    # 데이터 불러오기
    train_data = pd.read_csv(TRAIN_DATA_PATH).dropna(subset=['comment', 'context', 'label'])
    valid_data = pd.read_csv(VALID_DATA_PATH).dropna(subset=['comment', 'context', 'label'])
    test_data = pd.read_csv(TEST_DATA_PATH).dropna(subset=['comment', 'context', 'label'])

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model.to(device)

    is_roberta = "roberta" in model_name.lower()

    # 데이터 사전 토큰화
    print("데이터 토큰화 진행 중...")
    if use_context:
        train_texts, train_pairs = train_data['context'].tolist(), train_data['comment'].tolist()
        valid_texts, valid_pairs = valid_data['context'].tolist(), valid_data['comment'].tolist()
        test_texts, test_pairs = test_data['context'].tolist(), test_data['comment'].tolist()
    else:
        train_texts, train_pairs = train_data['comment'].tolist(), None
        valid_texts, valid_pairs = valid_data['comment'].tolist(), None
        test_texts, test_pairs = test_data['comment'].tolist(), None

    train_encodings = tokenizer(train_texts, train_pairs, truncation=True, padding="max_length", max_length=MAX_LENGTH, return_token_type_ids=not is_roberta)
    valid_encodings = tokenizer(valid_texts, valid_pairs, truncation=True, padding="max_length", max_length=MAX_LENGTH, return_token_type_ids=not is_roberta)
    test_encodings = tokenizer(test_texts, test_pairs, truncation=True, padding="max_length", max_length=MAX_LENGTH, return_token_type_ids=not is_roberta)

    train_dataset = HateSpeechDataset(train_encodings, train_data['label'].tolist())
    valid_dataset = HateSpeechDataset(valid_encodings, valid_data['label'].tolist())
    test_dataset = HateSpeechDataset(test_encodings, test_data['label'].tolist())

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # 클래스 가중치 계산
    classes = np.unique(train_data['label'])
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=train_data['label'].values)
    class_weights = torch.tensor(weights, dtype=torch.float).to(device)
    print("Class weights:", class_weights)

    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
    optimizer = AdamW(model.parameters(), lr=2e-5)
    
    gradient_accumulation_steps = 4 
    
    # 베스트 모델 저장을 위한 변수 초기화
    best_valid_f1 = 0.0
    best_model_state = None

    print("학습을 시작합니다...")
    for epoch in range(EPOCHS):
        model.train()
        total_train_loss = 0
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")

            outputs = model(**batch)
            loss = loss_fn(outputs.logits, labels)
            
            loss = loss / gradient_accumulation_steps
            loss.backward()
            total_train_loss += loss.item() * gradient_accumulation_steps

            if step % gradient_accumulation_steps == 0 or step == len(train_loader):
                optimizer.step()
                optimizer.zero_grad()

            if step % 150 == 0:
                print(f"Epoch {epoch + 1}/{EPOCHS} | Step {step}/{len(train_loader)} | Loss: {loss.item() * gradient_accumulation_steps:.4f}")

        avg_train_loss = total_train_loss / len(train_loader)
        valid_result = evaluate_model(model, valid_loader)

        print(f"\nEpoch {epoch + 1}/{EPOCHS} 결과")
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Valid Accuracy: {valid_result['accuracy']:.4f}")
        print(f"Valid Precision: {valid_result['precision']:.4f}")
        print(f"Valid Recall: {valid_result['recall']:.4f}")
        print(f"Valid F1: {valid_result['f1']:.4f}")

        # 베스트 모델 갱신 확인 로직
        if valid_result['f1'] > best_valid_f1:
            best_valid_f1 = valid_result['f1']
            best_model_state = copy.deepcopy(model.state_dict())
            print(f">>> 베스트 모델 갱신 (Valid F1: {best_valid_f1:.4f})")
        print("-" * 70)

    # 테스트 진행 전에 베스트 모델 로드
    print("\n가장 성능이 좋았던 모델을 불러와 테스트를 진행합니다...")
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        
    test_result = evaluate_model(model, test_loader)

    print("\n===== TEST RESULT (BEST EPOCH) =====")
    print(f"Accuracy : {test_result['accuracy']:.4f}")
    print(f"Precision: {test_result['precision']:.4f}")
    print(f"Recall   : {test_result['recall']:.4f}")
    print(f"F1-score : {test_result['f1']:.4f}")

    print("\n===== Classification Report =====")
    print(classification_report(test_result["y_true"], test_result["y_pred"], digits=4))

    # 메모리 정리
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "model": output_name,
        "accuracy": test_result["accuracy"],
        "precision": test_result["precision"],
        "recall": test_result["recall"],
        "f1": test_result["f1"]
    }

## 5. Baseline 1: BERT 기반 댓글 단독 분류 모델

In [ ]:
bert_result = train_baseline_model(
    model_name="klue/bert-base",
    output_name="BERT_comment_only"
)

bert_result

## 6. Baseline 2: RoBERTa 기반 댓글 단독 분류 모델

In [ ]:
roberta_result = train_baseline_model(
    model_name="klue/roberta-base",
    output_name="RoBERTa_comment_only"
)

roberta_result

## 7. Baseline 결과 비교표 저장

In [ ]:
result_df = pd.DataFrame([bert_result, roberta_result])

display(result_df)

result_df.to_csv("baseline_result.csv", index=False, encoding="utf-8-sig")

print("결과 저장 완료: baseline_result.csv")